# Modern Tokenization in LLMs — Hands-on Notebook

**Course:** Artificial Intelligence · **Lecturer:** Dr. Oybek Eraliev, Department of Computer Engineering, Inha University in Tashkent

This notebook implements, **from scratch in plain Python**, the four tokenization techniques from the lecture slides:

| # | Method | Direction | What decides the next step |
|---|--------|-----------|----------------------------|
| 1 | **Byte-Level BPE** | bottom-up (merge) | most frequent adjacent pair |
| 2 | **WordPiece** | bottom-up (merge) | `count(ab) / (count(a) · count(b))` |
| 3 | **Unigram LM** | top-down (prune) | smallest increase in corpus loss |
| 4 | **SentencePiece (BPE mode)** | bottom-up (merge) | most frequent pair, on raw text with `▁` |

All four use **the same training text** as the slides. For every method we print:

* the **vocabulary before iteration 1**,
* what happens **inside each iteration** (pair counts / scores / losses),
* the **vocabulary after each iteration**, with tokens marked as **added (+)** or **removed (−)**.

So you can watch how the vocabulary grows (BPE, WordPiece, SentencePiece) or shrinks (Unigram) step by step.

> **How to use:** run the cells from top to bottom (`Kernel → Restart & Run All`). Then change `TEXT` or the number of iterations and see what changes.

## 0. Setup: the running example and shared helpers

The text is intentionally tiny (16 words, 9 distinct letters) so every number can be checked by hand and compared with the slides.

In [1]:
import math
import re
import unicodedata
from collections import Counter

import pandas as pd
pd.set_option("display.max_colwidth", 80)

# The SAME training text that is used on every slide
TEXT = "hugs pugs news pugs nut hugs pugs pug nut pugs hugs news pugs nut hugs pugs"

word_counts = Counter(TEXT.split())
print("Number of words :", sum(word_counts.values()))
print("Word frequencies:", dict(word_counts.most_common()))

char_counts = Counter()
for w, f in word_counts.items():
    for ch in w:
        char_counts[ch] += f          # weighted by word frequency
print("Characters      :", dict(char_counts.most_common()))

Number of words : 16
Word frequencies: {'pugs': 6, 'hugs': 4, 'nut': 3, 'news': 2, 'pug': 1}
Characters      : {'u': 14, 's': 12, 'g': 11, 'p': 7, 'n': 5, 'h': 4, 't': 3, 'e': 2, 'w': 2}


### Shared helper: show a vocabulary and how it changed

`show_vocab()` is used by all four methods. It prints the current vocabulary and, if we pass the previous vocabulary, the tokens that were **added (+)** or **removed (−)** in this iteration.

In [2]:
def show_vocab(title, vocab, previous=None, order=None):
    """Print a vocabulary and (optionally) the difference to the previous one.

    vocab     : iterable of tokens (the current vocabulary)
    previous  : iterable of tokens (the vocabulary before this step) or None
    order     : optional list giving the display order (e.g. order of creation)
    """
    vocab = list(vocab)
    if order is not None:
        vocab = [t for t in order if t in set(vocab)]
    print(f"--- {title}  (|V| = {len(vocab)}) ---")
    print("   ", " ".join(f"[{t}]" for t in vocab))
    if previous is not None:
        prev = set(previous)
        added = [t for t in vocab if t not in prev]
        removed = [t for t in previous if t not in set(vocab)]
        if added:
            print("    + added  :", " ".join(f"[{t}]" for t in added))
        if removed:
            print("    − removed:", " ".join(f"[{t}]" for t in removed))
        if not added and not removed:
            print("    (no change)")
    print()


def show_corpus(splits, freqs, highlight=None):
    """Print every word type as a list of tokens, with its frequency.
    Tokens equal to `highlight` are shown in «» so the new token stands out."""
    rows = []
    for word, toks in splits.items():
        shown = " ".join(f"«{t}»" if t == highlight else t for t in toks)
        rows.append({"word type": word, "freq": freqs[word], "tokens": shown, "#tokens": len(toks)})
    display(pd.DataFrame(rows))

### Shared helpers for merge-based methods (BPE, WordPiece, SentencePiece-BPE)

Training data is stored as a dictionary **word type → list of symbols**, together with the frequency of each word type. Two small functions do all the work:

* `count_pairs()` — counts every adjacent pair `(a, b)`, **weighted by word frequency**;
* `apply_merge()` — replaces every occurrence of the pair `a b` by the new symbol.

In [3]:
def count_pairs(splits, freqs):
    """Count adjacent symbol pairs, weighted by how often each word occurs."""
    pair_counts = Counter()
    for word, symbols in splits.items():
        for a, b in zip(symbols, symbols[1:]):
            pair_counts[(a, b)] += freqs[word]
    return pair_counts


def count_units(splits, freqs):
    """Count single symbols (needed by WordPiece's score)."""
    unit_counts = Counter()
    for word, symbols in splits.items():
        for s in symbols:
            unit_counts[s] += freqs[word]
    return unit_counts


def apply_merge(splits, a, b, new_symbol):
    """Replace every adjacent 'a b' by new_symbol in every word (left to right)."""
    new_splits = {}
    for word, symbols in splits.items():
        out, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(new_symbol)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        new_splits[word] = out
    return new_splits

---
## 1. Byte-Level BPE (GPT-2 style)

**Idea.** Start from the 256 possible byte values, then repeatedly **merge the most frequent adjacent pair** into a new token. The ordered list of merges *is* the tokenizer.

### Step 0a — bytes and the `Ġ` symbol
GPT-2 maps each of the 256 bytes to a printable Unicode character, so that merges are readable. Most printable ASCII characters map to themselves; the space byte `0x20` becomes **`Ġ`** (U+0120).

In [4]:
def bytes_to_unicode():
    """GPT-2's reversible mapping: byte value (0..255) -> printable character."""
    printable = (list(range(ord("!"), ord("~") + 1)) +
                 list(range(ord("¡"), ord("¬") + 1)) +
                 list(range(ord("®"), ord("ÿ") + 1)))
    bs, cs, n = printable[:], printable[:], 0
    for b in range(256):
        if b not in bs:                 # non-printable bytes get shifted to 256+n
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}

BYTE2CHAR = bytes_to_unicode()

def to_byte_symbols(s):
    """String -> UTF-8 bytes -> GPT-2 printable characters."""
    return [BYTE2CHAR[b] for b in s.encode("utf-8")]

print("Base vocabulary size:", len(BYTE2CHAR), "byte symbols")
for s in ["hugs pugs", " café", " oʻzbek"]:
    raw = s.encode("utf-8")
    print(f"{s!r:12} bytes = {' '.join(f'{b:02X}' for b in raw):30} -> symbols = {to_byte_symbols(s)}")

Base vocabulary size: 256 byte symbols
'hugs pugs'  bytes = 68 75 67 73 20 70 75 67 73     -> symbols = ['h', 'u', 'g', 's', 'Ġ', 'p', 'u', 'g', 's']
' café'      bytes = 20 63 61 66 C3 A9              -> symbols = ['Ġ', 'c', 'a', 'f', 'Ã', '©']
' oʻzbek'    bytes = 20 6F CA BB 7A 62 65 6B        -> symbols = ['Ġ', 'o', 'Ê', '»', 'z', 'b', 'e', 'k']


### Step 0b — pre-tokenization
A regular expression splits the text into "words" that **keep their leading space**. Notice that the very first word, `hugs`, has no space in front of it, so it becomes a different word type than `Ġhugs`.

In [5]:
import regex  # supports \p{L} (any letter) like GPT-2's original pattern

GPT2_PATTERN = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

pieces = GPT2_PATTERN.findall(TEXT)
print("First pieces:", pieces[:6], "...")

# word type (written with Ġ) -> frequency
bbpe_freqs = Counter("".join(to_byte_symbols(p)) for p in pieces)
# word type -> list of byte symbols (the starting point of training)
bbpe_splits = {w: list(w) for w in bbpe_freqs}

show_corpus(bbpe_splits, bbpe_freqs)

First pieces: ['hugs', ' pugs', ' news', ' pugs', ' nut', ' hugs'] ...


,word type,freq,tokens,#tokens
0,hugs,1,h u g s,4
1,Ġpugs,6,Ġ p u g s,5
2,Ġnews,2,Ġ n e w s,5
3,Ġnut,3,Ġ n u t,4
4,Ġhugs,3,Ġ h u g s,5
5,Ġpug,1,Ġ p u g,4


### Step 1 — the training loop
In every iteration we:
1. count all adjacent pairs (weighted by word frequency),
2. pick the most frequent pair `(a, b)`,
3. add the new token `ab` to the vocabulary and record the merge rule,
4. rewrite the corpus.

The function prints everything that happens, including the **vocabulary before iteration 1** and **after every iteration**.

> Only 10 of the 256 byte symbols occur in our text, so we display those 10 as the "visible" base vocabulary (the other 246 are still in the vocabulary, they just never appear).

In [6]:
def train_bpe(splits, freqs, num_merges, top_k=8, label="BPE"):
    """Train BPE by repeatedly merging the most frequent pair. Returns (merges, vocab, splits)."""
    # Vocabulary before iteration 1 = all symbols that occur in the corpus
    vocab = []
    for symbols in splits.values():
        for s in symbols:
            if s not in vocab:
                vocab.append(s)
    show_vocab(f"{label}: vocabulary BEFORE iteration 1", vocab)

    merges = []
    for it in range(1, num_merges + 1):
        pair_counts = count_pairs(splits, freqs)
        ranked = pair_counts.most_common()          # ties: first pair encountered wins
        (a, b), best = ranked[0]

        print(f"================ Iteration {it} ================")
        table = pd.DataFrame([{"pair": f"{x} + {y}", "count": c} for (x, y), c in ranked[:top_k]])
        display(table)
        new_symbol = a + b
        print(f"Winner: '{a}' + '{b}' -> '{new_symbol}'   (count = {best})")

        splits = apply_merge(splits, a, b, new_symbol)
        merges.append((a, b))
        previous = list(vocab)
        vocab.append(new_symbol)

        show_corpus(splits, freqs, highlight=new_symbol)
        show_vocab(f"{label}: vocabulary AFTER iteration {it}", vocab, previous)
    return merges, vocab, splits

bbpe_merges, bbpe_vocab, bbpe_final = train_bpe(bbpe_splits, bbpe_freqs, num_merges=4, label="Byte-level BPE")

--- Byte-level BPE: vocabulary BEFORE iteration 1  (|V| = 10) ---
    [h] [u] [g] [s] [Ġ] [p] [n] [e] [w] [t]

================ Iteration 1 ================


,pair,count
0,u + g,11
1,g + s,10
2,Ġ + p,7
3,p + u,7
4,Ġ + n,5
5,h + u,4
6,n + u,3
7,u + t,3


Winner: 'u' + 'g' -> 'ug'   (count = 11)


,word type,freq,tokens,#tokens
0,hugs,1,h «ug» s,3
1,Ġpugs,6,Ġ p «ug» s,4
2,Ġnews,2,Ġ n e w s,5
3,Ġnut,3,Ġ n u t,4
4,Ġhugs,3,Ġ h «ug» s,4
5,Ġpug,1,Ġ p «ug»,3


--- Byte-level BPE: vocabulary AFTER iteration 1  (|V| = 11) ---
    [h] [u] [g] [s] [Ġ] [p] [n] [e] [w] [t] [ug]
    + added  : [ug]

================ Iteration 2 ================


,pair,count
0,ug + s,10
1,Ġ + p,7
2,p + ug,7
3,Ġ + n,5
4,h + ug,4
5,n + u,3
6,u + t,3
7,Ġ + h,3


... (7 outputs left out)

,pair,count
0,Ġp + ugs,6
1,Ġ + n,5
2,h + ugs,4
3,n + u,3
4,u + t,3
5,Ġ + h,3
6,n + e,2
7,e + w,2


Winner: 'Ġp' + 'ugs' -> 'Ġpugs'   (count = 6)


,word type,freq,tokens,#tokens
0,hugs,1,h ugs,2
1,Ġpugs,6,«Ġpugs»,1
2,Ġnews,2,Ġ n e w s,5
3,Ġnut,3,Ġ n u t,4
4,Ġhugs,3,Ġ h ugs,3
5,Ġpug,1,Ġp ug,2


--- Byte-level BPE: vocabulary AFTER iteration 4  (|V| = 14) ---
    [h] [u] [g] [s] [Ġ] [p] [n] [e] [w] [t] [ug] [ugs] [Ġp] [Ġpugs]
    + added  : [Ġpugs]



**What to notice**

| iteration | merge | why |
|---|---|---|
| 1 | `u + g → ug` | 6 (Ġpugs) + 3 (Ġhugs) + 1 (hugs) + 1 (Ġpug) = **11** |
| 2 | `ug + s → ugs` | 6 + 3 + 1 = **10** (Ġpug has no *s*) |
| 3 | `Ġ + p → Ġp` | 6 + 1 = **7**, beats `p + ugs` = 6 |
| 4 | `Ġp + ugs → Ġpugs` | the most frequent word becomes **one token** |

The vocabulary only **grows**: one new token per iteration.

### Step 2 — encoding new text
To encode, split into byte symbols and **replay the merge rules in the order they were learned**. Nothing can be unknown, because every string is a sequence of bytes.

In [ ]:
def bpe_encode_word(symbols, merges):
    """Apply learned merges, in training order, to one pre-tokenized word."""
    for a, b in merges:
        symbols = apply_merge({"w": symbols}, a, b, a + b)["w"]
    return symbols

def bbpe_encode(text, merges):
    tokens = []
    for piece in GPT2_PATTERN.findall(text):
        tokens += bpe_encode_word(to_byte_symbols(piece), merges)
    return tokens

for s in ["hugs pugs nut", " pug", " hugged", " café", " oʻzbek"]:
    print(f"{s!r:16} -> {bbpe_encode(s, bbpe_merges)}")

### (Optional) Check against the Hugging Face `tokenizers` library
If the library is installed (`pip install tokenizers`), training a real byte-level BPE on the same text gives **exactly the same merges**.

In [ ]:
!pip install tokenizers

In [ ]:
try:
    import json
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    hf = Tokenizer(models.BPE())
    hf.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    trainer = trainers.BpeTrainer(vocab_size=256 + 4, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    hf.train_from_iterator([TEXT], trainer)
    hf_merges = [tuple(m) if isinstance(m, list) else tuple(m.split()) for m in json.loads(hf.to_str())["model"]["merges"]]
    print("HF merges  :", hf_merges)
    print("Our merges :", bbpe_merges)
    print("Identical? :", hf_merges == bbpe_merges)
except ImportError:
    print("tokenizers is not installed - skip this check.")

---
## 2. WordPiece (BERT style)

**Idea.** Same merge loop as BPE, but the pair is chosen by a **score**, not by raw frequency:

$$\text{score}(a,b) = \frac{\text{count}(ab)}{\text{count}(a)\times\text{count}(b)}$$

A pair scores high when its parts **rarely appear apart** (similar to pointwise mutual information).
Text is split on whitespace first (no `Ġ`), and every symbol that is *not* at the start of a word gets the prefix **`##`**.

In [ ]:
wp_freqs = Counter(TEXT.split())            # "hugs" is the same type everywhere (x4)
wp_splits = {w: [w[0]] + ["##" + ch for ch in w[1:]] for w in wp_freqs}

show_corpus(wp_splits, wp_freqs)
print("Unit counts:", dict(count_units(wp_splits, wp_freqs)))

### Training loop with the WordPiece score
When two symbols are merged, the `##` of the second one is dropped: `n + ##ew → new`, `##e + ##w → ##ew`.

In [ ]:
def wp_join(a, b):
    """Join two WordPiece symbols: the continuation marker of b disappears."""
    return a + (b[2:] if b.startswith("##") else b)


def train_wordpiece(splits, freqs, num_merges, top_k=9):
    vocab = []
    for symbols in splits.values():
        for s in symbols:
            if s not in vocab:
                vocab.append(s)
    show_vocab("WordPiece: vocabulary BEFORE iteration 1", vocab)

    merges = []
    for it in range(1, num_merges + 1):
        pair_counts = count_pairs(splits, freqs)
        unit_counts = count_units(splits, freqs)

        rows = []
        for (a, b), c_ab in pair_counts.items():
            score = c_ab / (unit_counts[a] * unit_counts[b])
            rows.append({"pair": f"{a} + {b}", "c(ab)": c_ab, "c(a)": unit_counts[a],
                         "c(b)": unit_counts[b], "score": round(score, 4), "_pair": (a, b)})
        # sort by score (stable: ties keep first-seen order)
        rows.sort(key=lambda r: -r["score"])
        a, b = rows[0]["_pair"]
        new_symbol = wp_join(a, b)

        print(f"================ Iteration {it} ================")
        display(pd.DataFrame(rows[:top_k]).drop(columns="_pair"))
        print(f"Winner: '{a}' + '{b}' -> '{new_symbol}'   (score = {rows[0]['score']})")

        # For comparison: what plain BPE would have picked
        (fa, fb), fc = pair_counts.most_common(1)[0]
        print(f"(BPE would pick the most frequent pair '{fa} + {fb}' with count {fc})")

        splits = apply_merge(splits, a, b, new_symbol)
        merges.append((a, b))
        previous = list(vocab)
        vocab.append(new_symbol)

        show_corpus(splits, freqs, highlight=new_symbol)
        show_vocab(f"WordPiece: vocabulary AFTER iteration {it}", vocab, previous)
    return merges, vocab, splits

wp_merges, wp_vocab, wp_final = train_wordpiece(wp_splits, wp_freqs, num_merges=4)

**What to notice**

| iteration | merge | score |
|---|---|---|
| 1 | `##e + ##w → ##ew` | 2 / (2·2) = **0.5000** — rare, but always together |
| 2 | `n + ##ew → new` | 2 / (5·2) = **0.2000** |
| 3 | `new + ##s → news` | 2 / (2·12) = **0.0833** > `##g + ##s` = 10 / (11·12) = 0.0758 |
| 4 | `##g + ##s → ##gs` | 10 / (11·10) = **0.0909** (count of `##s` dropped to 10) |

BPE's first merges were **ug, ugs, Ġp, Ġpugs** (frequency); WordPiece's are **##ew, new, news, ##gs** (association).

### Inference: greedy longest-match-first
BERT stores only the vocabulary, not the merges. For each word, take the **longest prefix that is in the vocabulary**, then continue with `##` pieces. If nothing matches, the whole word becomes `[UNK]`.

In [ ]:
def wordpiece_encode_word(word, vocab, verbose=False):
    tokens, start = [], 0
    while start < len(word):
        end, piece = len(word), None
        while start < end:                       # try the longest candidate first
            cand = word[start:end] if start == 0 else "##" + word[start:end]
            if verbose:
                print(f"   try {cand!r:9} {'✓' if cand in vocab else '✗'}")
            if cand in vocab:
                piece = cand
                break
            end -= 1
        if piece is None:                        # no piece fits -> whole word unknown
            return ["[UNK]"]
        tokens.append(piece)
        start = end
    return tokens

print("Tracing 'hugs':")
print(" ->", wordpiece_encode_word("hugs", set(wp_vocab), verbose=True))
print()
for w in ["pugs", "newt", "nuts", "bug"]:
    print(f"{w!r:7} -> {wordpiece_encode_word(w, set(wp_vocab))}")

---
## 3. Unigram Language Model (Kudo, 2018)

**Idea.** Work in the opposite direction: start with a **large** vocabulary and **remove** the tokens the model needs least.

* Each token $t$ has a probability $p(t)$; a segmentation $x = (x_1,\dots,x_m)$ has probability $P(x)=\prod_i p(x_i)$.
* The tokenizer chooses the most probable segmentation $x^*$ with the **Viterbi** algorithm.
* Corpus loss: $L = \sum_w \text{freq}(w)\cdot\big(-\log P(x^*(w))\big)$.
* The **score** of a token is how much $L$ increases if we delete it: $\text{score}(t) = L(V\setminus\{t\}) - L(V)$.

> **Simplifications for hand calculation (as in the slides and the Hugging Face course):**
> the loss uses only the best segmentation, and probabilities come from substring frequencies, renormalised after each pruning step. Real SentencePiece re-estimates $p(t)$ with the EM algorithm.

### Seed vocabulary
All single characters, plus every substring that is **not a whole word** and occurs at least **4** times (weighted by word frequency).

In [ ]:
uni_words = Counter(TEXT.split())
MIN_FREQ = 4

# frequency of every substring (length >= 2) that is not a whole word
substr_freq = Counter()
for w, f in uni_words.items():
    for i in range(len(w)):
        for j in range(i + 2, len(w) + 1):
            if w[i:j] != w:
                substr_freq[w[i:j]] += f
print("All candidate substrings:", dict(substr_freq.most_common()))

uni_freq = dict(char_counts)                         # characters are always kept
uni_freq.update({t: f for t, f in substr_freq.items() if f >= MIN_FREQ})
CHARS = set(char_counts)

def probs(vocab):
    """p(t) = freq(t) / sum of freq over the current vocabulary."""
    total = sum(uni_freq[t] for t in vocab)
    return {t: uni_freq[t] / total for t in vocab}

seed_vocab = sorted(uni_freq, key=lambda t: (-uni_freq[t], t))
p0 = probs(seed_vocab)
display(pd.DataFrame([{"token": t, "freq": uni_freq[t], "p": round(p0[t], 4),
                       "-log p": round(-math.log(p0[t]), 3)} for t in seed_vocab]))
print("Sum of frequencies:", sum(uni_freq.values()))

### Viterbi segmentation and corpus loss
`best[i]` holds the cheapest way (smallest total $-\log p$) to segment the first `i` characters of the word.

In [ ]:
def viterbi(word, p):
    """Return (cost, tokens) of the most probable segmentation of `word`."""
    best = [(0.0, [])] + [(math.inf, None)] * len(word)
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in p and best[start][0] < math.inf:
                cost = best[start][0] - math.log(p[piece])
                if cost < best[end][0] - 1e-12:
                    best[end] = (cost, best[start][1] + [piece])
    return best[-1]


def corpus_loss(p, show=False):
    total, rows = 0.0, []
    for w, f in uni_words.items():
        cost, seg = viterbi(w, p)
        total += f * cost
        rows.append({"word": w, "freq": f, "best x*": " | ".join(seg),
                     "-log P": round(cost, 3), "freq × -log P": round(f * cost, 3)})
    if show:
        display(pd.DataFrame(rows))
        print(f"Corpus loss L = {total:.3f}")
    return total


def all_segmentations(word, vocab):
    """Enumerate every segmentation of `word` with tokens from `vocab` (small words only!)."""
    if not word:
        return [[]]
    out = []
    for j in range(1, len(word) + 1):
        if word[:j] in vocab:
            out += [[word[:j]] + rest for rest in all_segmentations(word[j:], vocab)]
    return out

# Look inside Viterbi for two words with the seed vocabulary
for w in ["hugs", "pugs"]:
    cands = sorted(all_segmentations(w, set(seed_vocab)), key=lambda s: sum(-math.log(p0[t]) for t in s))
    print(f"Candidates for {w!r} (best first):")
    for seg in cands[:4]:
        parts = " + ".join(f"{-math.log(p0[t]):.3f}" for t in seg)
        print(f"   {'|'.join(seg):10} {parts} = {sum(-math.log(p0[t]) for t in seg):.3f}")

### Training loop: score, prune, renormalise
Each iteration removes the **2** multi-character tokens with the **lowest score** (ties broken by lower frequency), until the vocabulary has **11** tokens. Single characters are never removed, so every word can always be segmented.

In [ ]:
def train_unigram(vocab, target_size, remove_per_iter=2):
    vocab = list(vocab)
    show_vocab("Unigram: vocabulary BEFORE iteration 1 (seed)", vocab)
    it = 0
    while len(vocab) > target_size:
        it += 1
        p = probs(vocab)
        print(f"================ Iteration {it} ================")
        print(f"Sum of frequencies = {sum(uni_freq[t] for t in vocab)}")
        L = corpus_loss(p, show=True)

        # score(t) = loss without t - loss with t  (probabilities NOT renormalised here)
        rows = []
        for t in vocab:
            if t in CHARS:
                continue
            p_without = {k: v for k, v in p.items() if k != t}
            rows.append({"token": t, "freq": uni_freq[t], "score": round(corpus_loss(p_without) - L, 3)})
        rows.sort(key=lambda r: (r["score"], r["freq"], r["token"]))
        display(pd.DataFrame(rows))

        k = min(remove_per_iter, len(vocab) - target_size)
        to_remove = [r["token"] for r in rows[:k]]
        print("Remove:", to_remove)

        previous = list(vocab)
        vocab = [t for t in vocab if t not in to_remove]
        show_vocab(f"Unigram: vocabulary AFTER iteration {it}", vocab, previous)
    return vocab

uni_vocab = train_unigram(seed_vocab, target_size=11)

**What to notice**

| iteration | loss L | removed | why |
|---|---|---|---|
| 1 | 109.364 | `hu`, `pu` | never used in a best segmentation → score 0 |
| 2 | 105.435 | `gs`, `ugs` | still unused; loss fell only because probabilities were renormalised |
| 3 | 97.050 | `ug` | `hug` (7.986) and `pug` (12.897) have become indispensable |

The vocabulary only **shrinks**. The survivors `hug` and `pug` are the stems; the suffix `s` stays separate.

In [ ]:
print("Final segmentation of the training words:")
p_final = probs(uni_vocab)
corpus_loss(p_final, show=True)

print("New words:")
for w in ["hug", "nuts", "pun"]:
    cost, seg = viterbi(w, p_final)
    print(f"   {w!r:7} -> {seg}   (-log P = {cost:.3f})")

### Subword regularization: sampling instead of Viterbi
During LLM training, Unigram can **sample** a segmentation with probability $\propto P(x)^\alpha$ (Kudo, 2018). The model then sees several segmentations of the same word, which makes it more robust. Below: the distribution for `hugs` with the seed (iteration-1) model, for three values of α.

In [ ]:
def sampling_distribution(word, vocab, alpha=1.0):
    p = probs(vocab)
    segs = all_segmentations(word, set(vocab))
    weights = [math.prod(p[t] for t in s) ** alpha for s in segs]
    Z = sum(weights)
    return sorted(((" | ".join(s), w / Z) for s, w in zip(segs, weights)), key=lambda x: -x[1])

table = {}
for alpha in [0.2, 1.0, 5.0]:
    for seg, share in sampling_distribution("hugs", seed_vocab, alpha):
        table.setdefault(seg, {})[f"alpha={alpha}"] = f"{100 * share:.1f}%"
display(pd.DataFrame(table).T)

import random
random.seed(0)
dist = sampling_distribution("hugs", seed_vocab, alpha=1.0)
samples = random.choices([s for s, _ in dist], weights=[w for _, w in dist], k=8)
print("8 random segmentations of 'hugs':", samples)

---
## 4. SentencePiece (BPE mode)

SentencePiece is a **toolkit**, not a new merge rule: it runs BPE or Unigram **directly on raw text**. What changes:

1. **NFKC normalisation** of Unicode;
2. every space becomes the visible meta symbol **`▁`** (U+2581), and a `▁` is also added in front of the first word (*dummy prefix*);
3. the base alphabet is **characters** (with optional *byte fallback* for unknown ones), not bytes.

Because spaces are kept as `▁`, decoding is **lossless**: join the pieces and turn `▁` back into spaces.

In [ ]:
SPACE = "▁"   # ▁

def sp_normalize(text):
    text = unicodedata.normalize("NFKC", text)
    return SPACE + text.replace(" ", SPACE)          # dummy prefix + visible spaces

normalized = sp_normalize(TEXT)
print("Normalised text:", normalized[:45], "...")

# split at ▁ boundaries (SentencePiece's default split_by_whitespace=true)
sp_pieces = [SPACE + w for w in normalized.split(SPACE) if w]
sp_freqs = Counter(sp_pieces)
sp_splits = {w: list(w) for w in sp_freqs}
show_corpus(sp_splits, sp_freqs)
print("Compare with byte-level BPE: there 'hugs' (x1) and 'Ġhugs' (x3) were two different types;")
print("here all four occurrences are the single type '▁hugs' (x4).")

### Training — the BPE loop from Section 1, applied to `▁`-marked characters

In [ ]:
sp_merges, sp_vocab, sp_final = train_bpe(sp_splits, sp_freqs, num_merges=4, label="SentencePiece-BPE")

### Encoding and lossless decoding

In [ ]:
def sp_encode(text, merges):
    tokens = []
    for w in [SPACE + w for w in sp_normalize(text).split(SPACE) if w]:
        tokens += bpe_encode_word(list(w), merges)
    return tokens

def sp_decode(tokens):
    return "".join(tokens).replace(SPACE, " ").lstrip(" ")   # remove the dummy prefix

for s in ["hugs pugs nut", "pug news"]:
    toks = sp_encode(s, sp_merges)
    print(f"{s!r:16} -> {toks}  -> decoded: {sp_decode(toks)!r}")

print()
print("Same text with byte-level BPE:", bbpe_encode("hugs pugs nut", bbpe_merges))

### Byte fallback
A character that never appeared in training (e.g. the Uzbek **ʻ**, U+02BB, as in *oʻzbek*) becomes `<unk>` — unless **byte fallback** is on, in which case it is written as its UTF-8 bytes.

In [ ]:
def sp_encode_with_fallback(text, merges, alphabet, byte_fallback=True):
    out = []
    for tok in sp_encode(text, merges):
        if len(tok) == 1 and tok not in alphabet:      # unseen single character
            if byte_fallback:
                out += [f"<0x{b:02X}>" for b in tok.encode("utf-8")]
            else:
                out.append("<unk>")
        else:
            out.append(tok)
    return out

alphabet = set(ch for w in sp_splits for ch in sp_splits[w])
# Our tiny corpus only knows the letters h u g s p n e w t, so we use a made-up word
# built from those letters plus the Uzbek ʻ (U+02BB): "pugʻ"
word = "pugʻ"
print(f"{word!r}: ʻ = U+{ord('ʻ'):04X} = UTF-8 bytes {' '.join(f'{b:02X}' for b in 'ʻ'.encode())}")
print("byte_fallback=False:", sp_encode_with_fallback(word, sp_merges, alphabet, byte_fallback=False))
print("byte_fallback=True :", sp_encode_with_fallback(word, sp_merges, alphabet, byte_fallback=True))

### (Optional) Check against the real `sentencepiece` library

In [ ]:
try:
    import os, tempfile
    import sentencepiece as spm
    tmp = tempfile.mkdtemp()
    with open(os.path.join(tmp, "corpus.txt"), "w", encoding="utf-8") as fh:
        fh.write(TEXT + "\n")
    spm.SentencePieceTrainer.train(input=os.path.join(tmp, "corpus.txt"), model_prefix=os.path.join(tmp, "sp"),
                                   model_type="bpe", vocab_size=3 + 10 + 4, hard_vocab_limit=False,
                                   character_coverage=1.0, minloglevel=2)
    sp = spm.SentencePieceProcessor(model_file=os.path.join(tmp, "sp.model"))
    learned = [sp.id_to_piece(i) for i in range(3, sp.get_piece_size())]
    print("sentencepiece pieces (merges first):", learned)
    print("our merged tokens                  :", [a + b for a, b in sp_merges])
    print("encode('hugs pugs nut')            :", sp.encode("hugs pugs nut", out_type=str))
except ImportError:
    print("sentencepiece is not installed - skip this check.")

---
## 5. Summary: the four vocabularies side by side

In [ ]:
def learned(vocab, base):
    return [t for t in vocab if t not in base]

summary = pd.DataFrame([
    {"method": "Byte-level BPE", "direction": "grow (+1 per iteration)",
     "start |V|": "256 (10 visible)", "final |V| (visible)": len(bbpe_vocab),
     "learned / kept multi-char tokens": ", ".join(learned(bbpe_vocab, set(sum(bbpe_splits.values(), []))))},
    {"method": "WordPiece", "direction": "grow (+1 per iteration)",
     "start |V|": len(wp_vocab) - len(wp_merges), "final |V| (visible)": len(wp_vocab),
     "learned / kept multi-char tokens": ", ".join(learned(wp_vocab, set(sum(wp_splits.values(), []))))},
    {"method": "Unigram LM", "direction": "shrink (−2 per iteration)",
     "start |V|": len(seed_vocab), "final |V| (visible)": len(uni_vocab),
     "learned / kept multi-char tokens": ", ".join(t for t in uni_vocab if t not in CHARS)},
    {"method": "SentencePiece-BPE", "direction": "grow (+1 per iteration)",
     "start |V|": len(sp_vocab) - len(sp_merges), "final |V| (visible)": len(sp_vocab),
     "learned / kept multi-char tokens": ", ".join(learned(sp_vocab, set(sum(sp_splits.values(), []))))},
])
display(summary)

print("Tokenizing the word 'hugs' with each method:")
print("  Byte-level BPE    :", bbpe_encode("hugs", bbpe_merges), "  /  ' hugs' ->", bbpe_encode(" hugs", bbpe_merges))
print("  WordPiece         :", wordpiece_encode_word("hugs", set(wp_vocab)))
print("  Unigram           :", viterbi("hugs", probs(uni_vocab))[1])
print("  SentencePiece-BPE :", sp_encode("hugs", sp_merges))

## 6. Exercises

1. **Byte-level BPE:** set `num_merges=6`. Which two merges are added after `Ġpugs`, and with what counts?
2. **WordPiece:** run 5 merges. Look at the score table of iteration 5 — what is special about it, and how does the code break the tie?
3. **Unigram:** change `MIN_FREQ` to 3 and `target_size` to 12. Which tokens survive now?
4. **SentencePiece:** compare the merge lists of Sections 1 and 4. Why are they the same, and where would they start to differ? *(Hint: look at the first word of `TEXT` and at the `Ġ` vs `▁` types.)*
5. **Your own text:** replace `TEXT` with one or two Uzbek sentences and rerun everything. Which method handles `oʻ` / `gʻ` best?